# AnnDataset Tutorial

This tutorial demonstrates the `AnnDataset` class - a PyTorch Dataset implementation for AnnData objects.

The `AnnDataset` class provides:

1. **Optional transform function**: Apply any custom preprocessing or augmentation to loaded data batches
2. **Memory efficiency**: Stream from backed data without loading into memory
3. **PyTorch integration**: Works seamlessly with DataLoaders and training loops
4. **Multiprocessing safety**: Robust handling of concurrent data access
5. **Optimized batch loading**: Sorts indices for efficient disk access

This makes it ideal for production ML workflows with AnnData objects!


In [ ]:
import sys
import os

# Add path to development version of anndata (if needed)
# sys.path.insert(0, os.path.abspath("../anndata/src"))

import numpy as np
import anndata as ad
import torch
from torch.utils.data import DataLoader
from anndata.experimental.pytorch import AnnDataset

print("✅ PyTorch is available")


## Create Demo Data

Let's create some synthetic single-cell data for demonstration purposes.


In [ ]:
# Create synthetic single-cell data
n_obs = 1000
n_vars = 2000

# Generate count matrix with realistic single-cell characteristics
np.random.seed(42)
X = np.random.negative_binomial(5, 0.3, (n_obs, n_vars)).astype(np.float32)

# Add some structure: make some cells have higher expression
high_expr_cells = np.random.choice(n_obs, n_obs // 3, replace=False)
X[high_expr_cells, :] *= 2

# Create metadata
obs = {
    'cell_type': np.random.choice(['T_cell', 'B_cell', 'NK_cell'], n_obs),
    'batch': np.random.choice(['batch_1', 'batch_2', 'batch_3'], n_obs),
    'total_counts': X.sum(axis=1)
}

var = {
    'gene_name': [f'Gene_{i:04d}' for i in range(n_vars)],
    'highly_variable': np.random.choice([True, False], n_vars, p=[0.1, 0.9])
}

# Create AnnData object
adata = ad.AnnData(X=X, obs=obs, var=var)
print(f"Created AnnData object: {adata}")


## Basic AnnDataset Usage

Let's create a basic dataset without any transforms.


In [ ]:
# Create basic dataset
dataset = AnnDataset(adata)

print(f"Dataset length: {len(dataset)}")
print(f"Dataset shape: {dataset.shape}")

# Get a single sample
sample = dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Expression data shape: {sample['X'].shape}")
print(f"Expression data type: {sample['X'].dtype}")


## Training Example

Let's create a realistic training example with preprocessing and augmentation combined in a single transform function.


In [ ]:
# Define a complete transform function for training
def training_transform(X):
    """Complete transform pipeline for training."""
    # Preprocessing: Normalize to 10,000 counts per cell
    row_sum = torch.sum(X) + 1e-8
    X = X * (1e4 / row_sum)

    # Preprocessing: Log1p transformation
    X = torch.log1p(X)

    # Augmentation: Add Gaussian noise for training
    noise = torch.randn_like(X) * 0.01
    X = X + noise

    return X

# Create dataset with transform
training_dataset = AnnDataset(
    adata,
    transform=training_transform,
    multiprocessing_safe=True,
    chunk_size=1000
)

# Test the transform
sample_raw = AnnDataset(adata)[0]['X']
sample_transformed = training_dataset[0]['X']

print(f"Raw data range: [{sample_raw.min():.2f}, {sample_raw.max():.2f}]")
print(f"Transformed data range: [{sample_transformed.min():.2f}, {sample_transformed.max():.2f}]")
print(f"Transform applied successfully!")
